# DL_04: Hybrid Probability Fusion, Validation Selection & Graph Integration
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements **Task 4: Hybrid Probability Fusion and Graph Integration** for the `alpha-sml/causal-nlp-reasoning` pipeline, integrating Phase 1 Applied Machine Learning predictions with Phase 2 Deep Learning predictions into a unified neuro-symbolic causal reasoning pipeline.

### Authoritative BioCause Model-Eligible Dataset Counts
In accordance with source reconstruction audits and strict token-budget limits (<= 384 tokens), the authoritative model-eligible split counts are:

| Split | Candidates | Positive (Causal = 1) | Negative (Non-Causal = 0) | Articles |
| :--- | ---:| ---:| ---:| ---:|
| **Train** | 9,928 | 621 | 9,307 | 12 PMC articles |
| **Validation** | 1,884 | 131 | 1,753 | 4 PMC articles |
| **Test** | 1,394 | 100 | 1,294 | 4 PMC articles |
| **Total Model-Eligible** | 13,206 | 852 | 12,354 | 20 PMC articles |

*(Note: 202 candidates from the 13,408 raw candidate pool were determined ineligible during Task 1 due to overlapping spans or token budget overflow and are tracked in `dl_biocause_reconstruction_diagnostics.csv`).*

---

## 2. Experimental Methodology & Theoretical Background

### Dual-Stream Neuro-Symbolic Architecture
1. **AML Stream (Symbolic / Feature-Engineered)**:
   - Evaluates syntactic dependency paths (SDP), lexical causal trigger cues (active, passive, discourse), positional distances, and TF-IDF n-grams.
   - Primary baseline: `Logistic Regression (Moderate Weight 1:2)` fitted on the training split (`data/processed/aml_biocause_*.csv`), providing strong structural precision.
2. **DL Stream (Neural Representation)**:
   - Sequence-level classification with `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` fine-tuned with special argument markers (`[E1]`, `[/E1]`, `[E2]`, `[/E2]`).
   - Deep contextual representations capture complex multi-clause syntax and implicit causal discourse (`data/processed/dl_biocause_*.csv`).

### Validation Discipline & Probability Fusion
- Continuous prediction probabilities are fused via linear interpolation:
  $$P_{\text{hybrid}} = \alpha \cdot P_{\text{DL}} + (1 - \alpha) \cdot P_{\text{AML}}$$
- **Strict Validation Selection**: The fusion parameter $\alpha \in [0.0, 1.0]$ (step 0.05) and decision threshold $\tau \in [0.10, 0.90]$ (step 0.05) are swept **strictly on the held-out validation split (1,884 instances)** to maximize validation Positive F1.
- **Frozen Evaluation**: Once $(\alpha^*, \tau^*)$ are selected, they are permanently frozen and evaluated **exactly once** on the held-out test split (1,394 instances). No test-set data influences hyperparameter selection.

### Symbolic Graph Compilation & Interventional Reasoning
- Surviving positive predictions are formatted into the canonical 17-column `UnifiedRelationRecord` schema (`dl_utils.UNIFIED_RELATION_SCHEMA`).
- Extracted relations are mapped to directed edges $(u \to v)$ in NetworkX.
- Deterministic cycle resolution (`enforce_dag_acyclicity` from `AML/causal_extractors.py`) prunes self-loops, resolves bidirectional conflicts, and breaks feedback cycles using edge confidence scores.
- Acyclic DAGs are compiled into `pgmpy.models.DiscreteBayesianNetwork` to illustrate Pearl's $do$-calculus versus observational conditioning.
- **Critical Guardrail**: DAG acyclicity is a mathematical property required for probabilistic inference, not empirical proof of causal ground truth. Corr2Cause benchmark data is strictly excluded from the BioCause knowledge graph.

In [1]:
# Section 1: Environment Setup, Portable Imports & Seed Initialization
import os
import sys
from pathlib import Path
import json
import re
from typing import Dict, Any, List, Tuple, Set

def find_repo_root():
    """Locate repository root across local, Colab, and Kaggle."""
    candidates = []
    curr = Path(os.getcwd()).resolve()
    candidates.extend([curr] + list(curr.parents))

    # Check Google Colab paths
    colab_candidates = [
        Path('/content/causal-nlp-reasoning'),
        Path('/content/drive/MyDrive/causal-nlp-reasoning'),
        Path('/content')
    ]
    candidates.extend(colab_candidates)

    # Check Kaggle paths
    candidates.extend([
        Path('/kaggle/working/causal-nlp-reasoning'),
        Path('/kaggle/working')
    ])

    for p in candidates:
        if p.exists() and (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
        if p.exists() and (p / 'dl_utils.py').exists():
            return p

    # Colab auto-clone fallback if repository files are not mounted
    if Path('/content').exists() and not (Path('/content/causal-nlp-reasoning') / 'DL' / 'dl_utils.py').exists():
        import subprocess
        print('Colab runtime detected without repository files. Cloning repository...')
        target = Path('/content/causal-nlp-reasoning')
        subprocess.run(['git', 'clone', 'https://github.com/alpha-sml/causal-nlp-reasoning.git', str(target)], check=False)
        if (target / 'DL' / 'dl_utils.py').exists():
            return target

    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
import dl_task4_utils
from causal_extractors import (
    enforce_dag_acyclicity,
    NEGATION_PATTERN,
)
import networkx as nx
import pandas as pd
import numpy as np
import pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)

# Seed determinism across python, numpy, torch
dl_utils.set_seed(42)

DATA_DIR = ROOT_DIR / 'data' / 'processed'
print("=" * 80)
print(f"REPOSITORY ROOT : {ROOT_DIR}")
rel_data_dir = DATA_DIR.relative_to(ROOT_DIR) if DATA_DIR.is_relative_to(ROOT_DIR) else DATA_DIR
print(f"DATA DIRECTORY  : {rel_data_dir}")
print(f"NETWORKX VERSION: {nx.__version__}")
print(f"PGMPY VERSION   : {pgmpy.__version__}")
print(f"PRIMARY AML CLF : {dl_utils.PRIMARY_AML_CLASSIFIER_NAME}")
print("=" * 80)
print("\nAuthoritative BioCause Model-Eligible Dataset Splits:")
for split, counts in dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS.items():
    print(f"  - {split.capitalize():<11}: Total = {counts['total']:,} | Positive = {counts['positive']:,} | Negative = {counts['negative']:,}")

/Users/Shubhamm/Documents/My_phone/causal-nlp-reasoning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/Shubhamm/Documents/My_phone/causal-nlp-reasoning/.venv/lib/python3.14/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


REPOSITORY ROOT : /Users/Shubhamm/Downloads/causal-nlp-reasoning
DATA DIRECTORY  : data/processed
NETWORKX VERSION: 3.7
PGMPY VERSION   : 1.1.2
PRIMARY AML CLF : Logistic Regression (Moderate Weight 1:2)

Authoritative BioCause Model-Eligible Dataset Splits:
  - Train      : Total = 9,928 | Positive = 621 | Negative = 9,307
  - Validation : Total = 1,884 | Positive = 131 | Negative = 1,753
  - Test       : Total = 1,394 | Positive = 100 | Negative = 1,294


## 3. Expected Inputs & Prediction Contracts

| Stream / Manifest | File Path | Expected Rows | Required Columns / Description |
| :--- | :--- | :---: | :--- |
| **AML Validation Preds** | `data/processed/aml_biocause_validation_predictions.csv` | 1,884 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **DL Validation Preds** | `data/processed/dl_biocause_validation_predictions.csv` | 1,884 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **Task 1 Val Split Manifest** | `data/processed/dl_biocause_val.csv` | 1,884 | Authoritative model-eligible validation candidate manifest |
| **AML Test Preds** | `data/processed/aml_biocause_test_predictions.csv` | 1,394 | `candidate_id, true_label, aml_prob, aml_pred, split` |
| **DL Test Preds** | `data/processed/dl_biocause_test_predictions.csv` | 1,394 | `candidate_id, true_label, dl_prob, dl_pred, split` |
| **Task 1 Test Split Manifest**| `data/processed/dl_biocause_test.csv` | 1,394 | Authoritative model-eligible test candidate manifest |
| **Candidate Metadata** | `data/processed/biocause_candidates.csv` | 13,408 | 17 metadata columns including `between_text` |
| **Gold Relations** | `data/processed/biocause_relations.csv` | 852 | Gold directed relation annotations (100 test) |

### Strict Execution Sequence & Evaluation Discipline
To guarantee zero data snooping or subtle hyperparameter leakage:
1. **Section 2**: Ingest and validate **validation predictions and isolated validation candidate metadata only**. Test metadata, test labels, test predictions, and gold relations are strictly NOT loaded or inspected during validation selection.
2. **Section 3**: Search and freeze ($\alpha^*, \tau^*$) strictly on validation data using the documented lexicographic rule (positive F1, then Macro-F1, then smaller $\alpha$, then smaller $\tau$).
3. **Section 4**: Ingest test predictions, isolated test candidate metadata, and gold test relations **only after** hyperparameter freezing, and evaluate exactly once.
4. **Pure Baselines**: Pure AML ($\tau = 0.50$) and Pure DL (documented $\tau = 0.70$ from Task 3 validation calibration) are evaluated with their fixed, previously established thresholds.

### Population Reference: Raw Candidates vs Model-Eligible Candidates
- **Raw Candidate Pool**: 13,408 total (Train: 10,086, Validation: 1,914, Test: 1,408).
- **Model-Eligible Population**: 13,206 total (Train: 9,928, Validation: 1,884, Test: 1,394).
- *202 candidates (Train: 158, Val: 30, Test: 14) were disqualified during Task 1 token-budget audits and are tracked in diagnostics.*

In [2]:
# Section 2: Validation Data Ingestion & Dual-Stream Alignment

def resolve_input_path(filename: str) -> Path:
    """Resolve prediction files across Local, Kaggle (/kaggle/working or /kaggle/input), and Colab."""
    candidates = [
        DATA_DIR / filename,
        ROOT_DIR / 'data' / 'processed' / filename,
        Path('/kaggle/working') / filename,
        Path('/kaggle/working/causal-nlp-reasoning/data/processed') / filename,
        Path('/content') / filename,
    ]
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob(filename):
            candidates.append(p)
    for c in candidates:
        if c.exists():
            return c
    return DATA_DIR / filename

# 1. Verify existence of ALL required files upfront (without parsing test contents before freeze)
aml_val_path = resolve_input_path("aml_biocause_validation_predictions.csv")
dl_val_path = resolve_input_path("dl_biocause_validation_predictions.csv")
aml_test_path = resolve_input_path("aml_biocause_test_predictions.csv")
dl_test_path = resolve_input_path("dl_biocause_test_predictions.csv")
val_manifest_path = resolve_input_path("dl_biocause_val.csv")
test_manifest_path = resolve_input_path("dl_biocause_test.csv")
cands_meta_path = resolve_input_path("biocause_candidates.csv")
relations_path = resolve_input_path("biocause_relations.csv")

required_files = {
    "AML Validation Predictions": aml_val_path,
    "DL Validation Predictions": dl_val_path,
    "AML Test Predictions": aml_test_path,
    "DL Test Predictions": dl_test_path,
    "Task 1 Prepared Val Split": val_manifest_path,
    "Task 1 Prepared Test Split": test_manifest_path,
    "BioCause Candidates Metadata": cands_meta_path,
    "BioCause Gold Relations": relations_path,
}

missing_files = [name for name, p in required_files.items() if not p.exists()]
if missing_files:
    missing_details = "\n".join(f"  - {name}: {required_files[name]}" for name in missing_files)
    raise FileNotFoundError(
        f"Missing required Task 4 input files:\n{missing_details}\n\n"
        "In strict adherence to task requirements, synthetic data must not be generated to "
        "substitute for missing model predictions or split manifests. If authoritative split files "
        "or DL prediction files from Task 3 are missing, Task 4 execution cannot proceed."
    )

print("All required input files verified on disk:")
for name, p in required_files.items():
    display_p = p.relative_to(ROOT_DIR) if p.is_relative_to(ROOT_DIR) else p
    row_count = sum(1 for _ in open(p)) - 1
    print(f"  ✓ {name:<30}: {display_p} ({row_count:,} rows)")

# 2. Ingest VALIDATION data and isolated validation metadata ONLY
# In strict adherence to audit rules: Test metadata, test labels, test predictions, and
# gold test relations are NOT loaded or retained in memory before the freeze.
print("\nIngesting validation predictions and isolated validation metadata...")
df_aml_val = pd.read_csv(aml_val_path)
df_dl_val = pd.read_csv(dl_val_path)

# Load authoritative model-eligible validation candidate manifest from Task 1 prepared split
# (dl_biocause_val.csv contains strictly validation records; zero test records or labels)
df_val_manifest = dl_task4_utils.validate_and_load_authoritative_manifest(
    val_manifest_path,
    "validation",
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["total"],
    expected_pos=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["positive"],
    expected_neg=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["negative"],
)
expected_val_ids = set(df_val_manifest["candidate_id"].astype(str))

# Validate validation predictions against contracts
dl_task4_utils.validate_prediction_dataframe(
    df_aml_val,
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["total"],
    "validation",
    "AML",
    "aml_prob",
    "aml_pred",
)
dl_task4_utils.validate_prediction_dataframe(
    df_dl_val,
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["total"],
    "validation",
    "DL",
    "dl_prob",
    "dl_pred",
)

# Validate cross-stream alignment on validation
dl_task4_utils.validate_dual_stream_alignment(df_aml_val, df_dl_val, "Validation")
print("  ✓ Validation: 100% candidate ID order alignment and 0 label discrepancies asserted.")

# Assert exact set equality between prediction IDs and authoritative Task 1 model-eligible IDs
dl_task4_utils.validate_exact_eligible_membership(df_aml_val["candidate_id"], expected_val_ids, "validation", "AML")
dl_task4_utils.validate_exact_eligible_membership(df_dl_val["candidate_id"], expected_val_ids, "validation", "DL")
print(f"  ✓ Validation: Exact set equality asserted with authoritative Task 1 model-eligible IDs ({len(expected_val_ids)} IDs).")

# Validate metadata split integrity and label agreement on validation
dl_task4_utils.validate_metadata_split_and_label_integrity(df_aml_val, df_val_manifest, "validation")
dl_task4_utils.validate_metadata_split_and_label_integrity(df_dl_val, df_val_manifest, "validation")
print("  ✓ Validation: Metadata split integrity & ground-truth label agreement validated.")

# 3. Assert strict isolation: No test metadata, predictions, or gold annotations loaded
assert (df_val_manifest["split"] == "validation").all(), "Leaked non-validation metadata!"
assert "test" not in df_val_manifest["split"].unique(), "Test candidates found in validation metadata!"
assert "df_cands_test_meta" not in globals(), "Test metadata must not be loaded during validation selection!"
assert "df_aml_test" not in globals(), "Test AML data must not be loaded during validation selection!"
assert "df_dl_test" not in globals(), "Test DL data must not be loaded during validation selection!"
assert "df_gold_relations" not in globals(), "Gold relations must not be loaded during validation selection!"
assert "df_cands_val_meta" not in globals(), "Combined candidate metadata must not be parsed before freeze!"
print("\n✓ Strict validation isolation asserted: Test metadata, labels, and gold relations are untouched.")

All required input files verified on disk:
  ✓ AML Validation Predictions    : data/processed/aml_biocause_validation_predictions.csv (1,884 rows)
  ✓ DL Validation Predictions     : data/processed/dl_biocause_validation_predictions.csv (1,884 rows)
  ✓ AML Test Predictions          : data/processed/aml_biocause_test_predictions.csv (1,394 rows)
  ✓ DL Test Predictions           : data/processed/dl_biocause_test_predictions.csv (1,394 rows)
  ✓ Task 1 Prepared Val Split     : data/processed/dl_biocause_val.csv (16,684 rows)
  ✓ Task 1 Prepared Test Split    : data/processed/dl_biocause_test.csv (11,420 rows)
  ✓ BioCause Candidates Metadata  : data/processed/biocause_candidates.csv (68,548 rows)
  ✓ BioCause Gold Relations       : data/processed/biocause_relations.csv (852 rows)

Ingesting validation predictions and isolated validation metadata...
  ✓ Validation: 100% candidate ID order alignment and 0 label discrepancies asserted.
  ✓ Validation: Exact set equality asserted with autho

## 4. Validation-Selected Probability-Score Fusion

### Linear Fusion Formula
For every candidate $i$, continuous probability estimates are combined via:
$$P_{\text{hybrid}}^{(i)} = \alpha \cdot P_{\text{DL}}^{(i)} + (1 - \alpha) \cdot P_{\text{AML}}^{(i)}$$

### Hyperparameter Search Protocol
- **Evaluation Discipline**: Model parameters $(\alpha, \tau)$ are tuned strictly on the **validation split (1,884 candidates)**. No test data is touched during tuning.
- **$\alpha$ Sweep**: Grid $\alpha \in [0.0, 1.0]$ in increments of 0.05 (21 configurations), spanning pure AML ($\alpha=0.0$), balanced ensembles ($\alpha=0.50$), and pure DL ($\alpha=1.0$).
- **Threshold $\tau$ Sweep**: Grid $\tau \in [0.10, 0.90]$ in increments of 0.05.
- **Primary Optimization Objective**: Maximize Validation Positive Causal F1 ($F_{1, \text{pos}}$), with Macro-F1 as secondary tie-breaker.
- **Freezing**: Once optimal $(\alpha^*, \tau^*)$ are identified, they are frozen for test evaluation.

In [3]:
# Section 3: Validation-Selected Probability-Score Fusion

val_y_true = df_aml_val['true_label'].values
val_p_aml = df_aml_val['aml_prob'].values
val_p_dl = df_dl_val['dl_prob'].values

print("Executing validation grid search with documented lexicographic selection rule:")
print("  1. Maximize positive-class F1.")
print("  2. If equal within tolerance (1e-5), maximize Macro-F1.")
print("  3. If both equal within tolerance, deterministic tie-break: smaller alpha, then smaller tau.")

best_alpha, best_tau, best_val_f1_pos, best_val_macro_f1, search_records = (
    dl_task4_utils.select_validation_fusion_hyperparameters(
        val_y_true=val_y_true,
        val_p_aml=val_p_aml,
        val_p_dl=val_p_dl,
        alpha_grid=np.round(np.arange(0.0, 1.05, 0.05), 2),
        tau_grid=np.round(np.arange(0.10, 0.95, 0.05), 2),
        rtol=1e-5,
        atol=1e-5,
    )
)

df_search = pd.DataFrame(search_records)

print("\nTop 5 Validation Hyperparameter Search Candidates:")
df_top5 = df_search.sort_values(by=['pos_f1', 'macro_f1', 'alpha', 'tau'], ascending=[False, False, True, True]).head(5)
print(df_top5.to_string(index=False))

# Freeze optimal hyperparameters permanently
FROZEN_ALPHA = float(best_alpha)
FROZEN_TAU = float(best_tau)

print("=" * 80)
print("OPTIMAL VALIDATION HYBRID CONFIGURATION (PERMANENTLY FROZEN)")
print("=" * 80)
print(f"Selected alpha*         : {FROZEN_ALPHA:.2f}")
print(f"Selected threshold tau*  : {FROZEN_TAU:.2f}")
print(f"Validation Positive F1  : {best_val_f1_pos:.4f}")
print(f"Validation Macro F1     : {best_val_macro_f1:.4f}")
print("=" * 80)
print("✓ Hyperparameters (alpha*, tau*) are now permanently frozen.")
print("  Test data has NOT been loaded, inspected, or used in any decision.")

# Save validation best fused probabilities for export
val_p_hybrid_best = dl_utils.fuse_probabilities(prob_aml=val_p_aml, prob_dl=val_p_dl, alpha=FROZEN_ALPHA)

Executing validation grid search with documented lexicographic selection rule:
  1. Maximize positive-class F1.
  2. If equal within tolerance (1e-5), maximize Macro-F1.
  3. If both equal within tolerance, deterministic tie-break: smaller alpha, then smaller tau.



Top 5 Validation Hyperparameter Search Candidates:
 alpha  tau  pos_precision  pos_recall   pos_f1  macro_f1
  0.00 0.35       0.380682     0.51145 0.436482  0.693248
  0.05 0.35       0.380682     0.51145 0.436482  0.693248
  0.10 0.35       0.380682     0.51145 0.436482  0.693248
  0.15 0.35       0.380682     0.51145 0.436482  0.693248
  0.20 0.35       0.380682     0.51145 0.436482  0.693248
OPTIMAL VALIDATION HYBRID CONFIGURATION (PERMANENTLY FROZEN)
Selected alpha*         : 0.00
Selected threshold tau*  : 0.35
Validation Positive F1  : 0.4365
Validation Macro F1     : 0.6932
✓ Hyperparameters (alpha*, tau*) are now permanently frozen.
  Test data has NOT been loaded, inspected, or used in any decision.


## 5. Held-Out Test Evaluation (Single Evaluation Run)

With $(\alpha^*, \tau^*)$ frozen strictly from validation data, we evaluate once on the held-out test split (1,394 candidates, 100 gold positive causal relations).

We contrast:
1. **Pure AML Baseline**: Logistic Regression (Moderate Weight 1:2)
2. **Pure PubMedBERT Baseline**: Task 3 fine-tuned model
3. **Selected Hybrid Model**: Linear probability fusion with frozen parameters

In [4]:
# Section 4: Held-Out Test Data Ingestion & Frozen Evaluation

# 1. Ingest test predictions, isolated test metadata, and gold relations ONLY NOW (after hyperparameter freezing)
print("Ingesting held-out test predictions (N = 1,394), test metadata, and gold relations...")
df_aml_test = pd.read_csv(aml_test_path)
df_dl_test = pd.read_csv(dl_test_path)
df_gold_relations = pd.read_csv(relations_path, keep_default_na=False)
df_cands_test_meta = pd.read_csv(cands_meta_path, keep_default_na=False).query("split == 'test'").copy()

# Load authoritative model-eligible test candidate manifest from Task 1 prepared split
df_test_manifest = dl_task4_utils.validate_and_load_authoritative_manifest(
    test_manifest_path,
    "test",
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["test"]["total"],
    expected_pos=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["test"]["positive"],
    expected_neg=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["test"]["negative"],
)
expected_test_ids = set(df_test_manifest["candidate_id"].astype(str))

# Validate test predictions against contracts
dl_task4_utils.validate_prediction_dataframe(
    df_aml_test,
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'],
    'test',
    'AML',
    'aml_prob',
    'aml_pred',
)
dl_task4_utils.validate_prediction_dataframe(
    df_dl_test,
    dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'],
    'test',
    'DL',
    'dl_prob',
    'dl_pred',
)

# Validate cross-stream alignment on test
dl_task4_utils.validate_dual_stream_alignment(df_aml_test, df_dl_test, 'Test')
print("  ✓ Test: 100% candidate ID order alignment and 0 label discrepancies asserted.")

# Assert exact set equality between test prediction IDs and authoritative Task 1 model-eligible IDs
dl_task4_utils.validate_exact_eligible_membership(df_aml_test["candidate_id"], expected_test_ids, "test", "AML")
dl_task4_utils.validate_exact_eligible_membership(df_dl_test["candidate_id"], expected_test_ids, "test", "DL")
print(f"  ✓ Test: Exact set equality asserted with authoritative Task 1 model-eligible IDs ({len(expected_test_ids)} IDs).")

# Validate metadata split integrity and label agreement on test
dl_task4_utils.validate_metadata_split_and_label_integrity(df_aml_test, df_test_manifest, "test")
dl_task4_utils.validate_metadata_split_and_label_integrity(df_dl_test, df_test_manifest, "test")
print("  ✓ Test: Metadata split integrity & ground-truth label agreement validated.")

test_y_true = df_aml_test['true_label'].values
test_p_aml = df_aml_test['aml_prob'].values
test_p_dl = df_dl_test['dl_prob'].values

# 2. Fuse test probabilities using FROZEN alpha*
test_p_hybrid = dl_utils.fuse_probabilities(prob_aml=test_p_aml, prob_dl=test_p_dl, alpha=FROZEN_ALPHA)
test_y_pred_hybrid = (test_p_hybrid >= FROZEN_TAU).astype(int)

# 3. Documented Baseline Thresholds:
# AML: Logistic Regression documented threshold = 0.50
# DL: Task 3 fine-tuned PubMedBERT calibrated threshold = 0.70 (and 0.50 default for reference)
# Do NOT tune baseline thresholds on test!
AML_DOCUMENTED_TAU = 0.50
DL_DOCUMENTED_TAU = 0.70  # Calibrated on Task 3 validation

def evaluate_stream(y_true: np.ndarray, y_prob: np.ndarray, threshold: float, stream_name: str) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec_bin, rec_bin, f1_bin, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    prec_mac, rec_mac, f1_mac, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    pr_auc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    return {
        'Stream': stream_name,
        'Threshold': threshold,
        'Accuracy': acc,
        'Pos Precision': prec_bin,
        'Pos Recall': rec_bin,
        'Pos F1': f1_bin,
        'Macro Precision': prec_mac,
        'Macro Recall': rec_mac,
        'Macro F1': f1_mac,
        'ROC-AUC': roc,
        'PR-AUC': pr_auc,
        'Confusion Matrix': cm,
    }

eval_aml = evaluate_stream(test_y_true, test_p_aml, AML_DOCUMENTED_TAU, "Pure AML (Logistic Regression, tau=0.50)")
eval_dl_documented = evaluate_stream(test_y_true, test_p_dl, DL_DOCUMENTED_TAU, "Pure DL (PubMedBERT, documented tau=0.70)")
eval_dl_default = evaluate_stream(test_y_true, test_p_dl, 0.50, "Pure DL (PubMedBERT, default tau=0.50)")
eval_hybrid_selected = evaluate_stream(test_y_true, test_p_hybrid, FROZEN_TAU, f"Hybrid Model (FROZEN alpha={FROZEN_ALPHA:.2f}, tau={FROZEN_TAU:.2f})")

assert eval_hybrid_selected['Stream'].startswith("Hybrid Model"), "Selected hybrid model identity mismatch!"

test_summary = [
    eval_aml,
    eval_dl_documented,
    eval_dl_default,
    eval_hybrid_selected,
]

df_test_eval = pd.DataFrame(test_summary)
print("\n" + "=" * 80)
print("HELD-OUT TEST SET EVALUATION SUMMARY (SINGLE RUN, FROZEN HYPERPARAMETERS)")
print("=" * 80)
print(df_test_eval[['Stream', 'Threshold', 'Pos Precision', 'Pos Recall', 'Pos F1', 'Macro F1', 'ROC-AUC', 'PR-AUC']].to_string(index=False))

print("\nConfusion Matrices [TN, FP / FN, TP]:")
for s in test_summary:
    cm = s['Confusion Matrix']
    print(f"  {s['Stream']:<48}: TN={cm[0,0]:>4}, FP={cm[0,1]:>3} | FN={cm[1,0]:>3}, TP={cm[1,1]:>3}")

Ingesting held-out test predictions (N = 1,394), test metadata, and gold relations...
  ✓ Test: 100% candidate ID order alignment and 0 label discrepancies asserted.
  ✓ Test: Exact set equality asserted with authoritative Task 1 model-eligible IDs (1394 IDs).
  ✓ Test: Metadata split integrity & ground-truth label agreement validated.

HELD-OUT TEST SET EVALUATION SUMMARY (SINGLE RUN, FROZEN HYPERPARAMETERS)
                                    Stream  Threshold  Pos Precision  Pos Recall   Pos F1  Macro F1  ROC-AUC   PR-AUC
  Pure AML (Logistic Regression, tau=0.50)       0.50       0.402439        0.33 0.362637  0.659062 0.910811 0.350029
 Pure DL (PubMedBERT, documented tau=0.70)       0.70       0.428571        0.06 0.105263  0.533559 0.910811 0.350029
    Pure DL (PubMedBERT, default tau=0.50)       0.50       0.402439        0.33 0.362637  0.659062 0.910811 0.350029
Hybrid Model (FROZEN alpha=0.00, tau=0.35)       0.35       0.373016        0.47 0.415929  0.682203 0.910811 0.3500

## 6. Linguistic Sanity Validation & Candidate Pruning

Even with strong neural-statistical confidence, candidate predictions can violate domain-specific linguistic invariants:
1. **Explicit Negation Filter**:
   - **Rule**: If the intervening text between cause and effect contains explicit negation cues matching `NEGATION_PATTERN` (e.g., `not`, `failed to`, `unable to`, `neither/nor`), the candidate relation is negated or inverted.
   - **Rationale**: Biological causal extraction seeks affirmative directed regulatory relationships. Negated assertions must not enter the knowledge graph as positive regulatory edges.
2. **Argument Distance Window Constraint**:
   - **Rule**: If the intervening text length exceeds 400 characters (`len(between_text) > 400`), the candidate is pruned.
   - **Rationale**: Long-range syntactic dependencies across multi-sentence or distal contexts suffer from parse degradation and false-positive entity pairing.

In [5]:
# Section 5: Linguistic Sanity Validation & Candidate Pruning

# Join test predictions with isolated test candidate metadata
df_test_full = df_aml_test[['candidate_id', 'true_label']].copy()
df_test_full['aml_prob'] = test_p_aml
df_test_full['dl_prob'] = test_p_dl
df_test_full['hybrid_prob'] = test_p_hybrid
df_test_full['raw_hybrid_pred'] = test_y_pred_hybrid
df_test_full['split'] = 'test'

# Join metadata fields from df_cands_test_meta (inner join with assertions to prevent row drops/duplications)
meta_cols = [
    'article_id', 'section_id', 'arg1_id', 'arg1_text', 'arg1_start', 'arg1_end',
    'arg2_id', 'arg2_text', 'arg2_start', 'arg2_end', 'char_distance', 'arg1_before_arg2', 'between_text'
]
df_test_full = df_test_full.merge(df_cands_test_meta[['candidate_id'] + meta_cols], on='candidate_id', how='inner')
assert len(df_test_full) == len(df_aml_test), f"Metadata merge row count mismatch: {len(df_test_full)} vs {len(df_aml_test)}"

# Apply linguistic filtering with auditable diagnostics and fail-fast missing data policy
MAX_BETWEEN_CHARS = 400
df_test_full = dl_task4_utils.apply_linguistic_filters(df_test_full, max_between_chars=MAX_BETWEEN_CHARS, pred_col='raw_hybrid_pred')

# Diagnostic reporting: exact non-overlapping categories
diag_counts = dl_task4_utils.compute_linguistic_diagnostics(df_test_full, pred_col='raw_hybrid_pred', sanitized_col='sanitized_pred')
n_initial_pos = diag_counts["initial_positive"]
n_pos_neg_only = diag_counts["rejected_negation_only"]
n_pos_dist_only = diag_counts["rejected_distance_only"]
n_pos_both = diag_counts["rejected_both"]
n_total_rejected = diag_counts["total_unique_rejected"]
n_sanitized_pos = diag_counts["surviving_positive"]

print("=" * 80)
print("LINGUISTIC SANITY FILTERING AUDIT (Held-Out Test Positive Predictions)")
print("=" * 80)
print(f"Initial raw positive predictions          : {n_initial_pos}")
print(f"  - Rejected by negation ONLY             : {n_pos_neg_only}")
print(f"  - Rejected by distance (>400) ONLY      : {n_pos_dist_only}")
print(f"  - Rejected by BOTH negation & distance  : {n_pos_both}")
print(f"Total unique positive candidates rejected : {n_total_rejected}")
print(f"Surviving sanitized positive relations    : {n_sanitized_pos}")
print(f"Audit verification: {n_sanitized_pos} + {n_pos_neg_only} + {n_pos_dist_only} + {n_pos_both} == {n_initial_pos} (Exact)")

# Evaluate precision/recall impact of linguistic filtering on test
prec_raw, rec_raw, f1_raw, _ = precision_recall_fscore_support(test_y_true, df_test_full['raw_hybrid_pred'], average='binary', zero_division=0)
prec_san, rec_san, f1_san, _ = precision_recall_fscore_support(test_y_true, df_test_full['sanitized_pred'], average='binary', zero_division=0)

print(f"\nImpact on Test Positive Causal (1) Metrics:")
print(f"  Raw Hybrid       : Precision = {prec_raw:.4f} | Recall = {rec_raw:.4f} | F1 = {f1_raw:.4f}")
print(f"  Sanitized Hybrid : Precision = {prec_san:.4f} | Recall = {rec_san:.4f} | F1 = {f1_san:.4f}")

LINGUISTIC SANITY FILTERING AUDIT (Held-Out Test Positive Predictions)
Initial raw positive predictions          : 126
  - Rejected by negation ONLY             : 12
  - Rejected by distance (>400) ONLY      : 0
  - Rejected by BOTH negation & distance  : 0
Total unique positive candidates rejected : 12
Surviving sanitized positive relations    : 114
Audit verification: 114 + 12 + 0 + 0 == 126 (Exact)

Impact on Test Positive Causal (1) Metrics:
  Raw Hybrid       : Precision = 0.3730 | Recall = 0.4700 | F1 = 0.4159
  Sanitized Hybrid : Precision = 0.3860 | Recall = 0.4400 | F1 = 0.4112


## 7. Standardized 17-Column UnifiedRelationRecord Formatting

Positive relations surviving linguistic filtering are formatted into the canonical `UnifiedRelationRecord` schema:
- Exactly 17 columns in specified order (`dl_utils.UNIFIED_RELATION_SCHEMA`).
- Stable relation ID generated via SHA-256 hash digest (`dl_utils.generate_relation_id`).
- Direction normalized strictly to `cause->effect`.

In [6]:
# Section 6: Standardized 17-Column UnifiedRelationRecord Formatting

df_pos_relations = df_test_full[df_test_full['sanitized_pred'] == 1].copy()

unified_records = []
candidate_id_to_relation_id = {}

for _, r in df_pos_relations.iterrows():
    record = dl_utils.build_unified_relation_record(
        candidate_id=str(r['candidate_id']),
        article_id=str(r['article_id']),
        section_id=str(r['section_id']),
        cause_id=str(r['arg1_id']),
        cause_text=str(r['arg1_text']),
        cause_start=int(r['arg1_start']),
        cause_end=int(r['arg1_end']),
        effect_id=str(r['arg2_id']),
        effect_text=str(r['arg2_text']),
        effect_start=int(r['arg2_start']),
        effect_end=int(r['arg2_end']),
        confidence=float(r['hybrid_prob']),
        extraction_method="hybrid_probability_fusion",
        evidence_text=str(r['between_text']),
        split="test",
        direction="cause->effect",
        provenance=f"PubMedBERT_alpha{FROZEN_ALPHA:.2f}+LogisticRegression_1to2_tau{FROZEN_TAU:.2f}",
    )
    unified_records.append(record)
    candidate_id_to_relation_id[str(r['candidate_id'])] = record['relation_id']

df_unified_records = pd.DataFrame(unified_records)

# Assert strict schema conformance
assert list(df_unified_records.columns) == dl_utils.UNIFIED_RELATION_SCHEMA, "UnifiedRelationRecord schema mismatch!"
assert len(df_unified_records) == n_sanitized_pos, "Record count mismatch with sanitized positives!"
assert (df_unified_records['direction'] == 'cause->effect').all(), "Non-directed relation found!"
assert (df_unified_records['split'] == 'test').all(), "Split mismatch in relation records!"

print("=" * 80)
print(f"STANDARDIZED RELATION RECORDS FORMATTING: {len(df_unified_records)} relations")
print("=" * 80)
print(f"Schema columns ({len(dl_utils.UNIFIED_RELATION_SCHEMA)}): {dl_utils.UNIFIED_RELATION_SCHEMA}")
print(f"All records mapped with candidate_id provenance and directionality asserted.")

STANDARDIZED RELATION RECORDS FORMATTING: 114 relations
Schema columns (17): ['relation_id', 'article_id', 'section_id', 'cause_id', 'cause_text', 'cause_start', 'cause_end', 'effect_id', 'effect_text', 'effect_start', 'effect_end', 'direction', 'confidence', 'extraction_method', 'evidence_text', 'provenance', 'split']
All records mapped with candidate_id provenance and directionality asserted.


## 8. NetworkX DAG Construction & Deterministic Cycle Breaking

### Causal Graph Formulation
- **Nodes**: Standardized biomedical entity identifiers qualified by section: `(section_id, entity_id)`.
- **Edges**: Directed causal edges $(u \to v)$ weighted by hybrid confidence $P_{\text{hybrid}}$.
- **Acyclicity Enforcement**: Reuses Phase 1 deterministic cycle breaker (`AML/causal_extractors.py` -> `enforce_dag_acyclicity`):
  1. Identifies and prunes self-loops $(u \to u)$.
  2. Resolves bidirectional conflicts $(u \rightleftharpoons v)$ by pruning the lower-confidence edge (with deterministic tie-breaking).
  3. Breaks multi-node feedback cycles by removing minimum-confidence feedback edges.

### Edge-Level Ground Truth Evaluation
We evaluate directed edge precision, recall, and F1 before and after cycle pruning against gold relations from `data/processed/biocause_relations.csv`.

In [7]:
# Section 7: NetworkX DAG Construction & Deterministic Cycle Breaking

# Build directed graph from positive relation records
G_pred = nx.DiGraph()
for _, r in df_unified_records.iterrows():
    u = f"{r['section_id']}_{r['cause_id']}"
    v = f"{r['section_id']}_{r['effect_id']}"
    if not G_pred.has_node(u):
        G_pred.add_node(u, display_text=r['cause_text'], section_id=r['section_id'])
    if not G_pred.has_node(v):
        G_pred.add_node(v, display_text=r['effect_text'], section_id=r['section_id'])
    if G_pred.has_edge(u, v):
        G_pred[u][v]['evidence_count'] += 1
        G_pred[u][v]['confidence'] = max(G_pred[u][v]['confidence'], float(r['confidence']))
    else:
        G_pred.add_edge(u, v, confidence=float(r['confidence']), evidence_count=1, extraction_method=r['extraction_method'])

n_nodes_before = G_pred.number_of_nodes()
n_edges_before = G_pred.number_of_edges()
is_dag_before = nx.is_directed_acyclic_graph(G_pred)

print(f"--- Initial Graph Topology ---")
print(f"  Nodes : {n_nodes_before}")
print(f"  Edges : {n_edges_before}")
print(f"  Is DAG: {is_dag_before}")

# Execute deterministic DAG enforcement
G_dag, audit_log = enforce_dag_acyclicity(G_pred, weight_attr="confidence")

n_nodes_after = G_dag.number_of_nodes()
n_edges_after = G_dag.number_of_edges()
is_dag_after = nx.is_directed_acyclic_graph(G_dag)

assert is_dag_after, "DAG enforcement failed to guarantee acyclicity!"

# Audit log breakdown
n_self_loops = sum(1 for e in audit_log if e.get('reason') == 'self_loop')
n_bidirectional = sum(1 for e in audit_log if 'bidirectional' in e.get('reason', ''))
n_cycle_feedback = sum(1 for e in audit_log if 'cycle' in e.get('reason', ''))

print(f"\n--- Cycle Breaking Diagnostics & Audit Log ---")
print(f"  Self-loops pruned             : {n_self_loops}")
print(f"  Bidirectional conflicts pruned: {n_bidirectional}")
print(f"  Feedback cycle edges pruned   : {n_cycle_feedback}")
print(f"  Total edges removed           : {len(audit_log)}")
print(f"  Final DAG Nodes               : {n_nodes_after}")
print(f"  Final DAG Edges               : {n_edges_after}")
print(f"  Acyclicity Guaranteed         : {is_dag_after}")

# Edge-level evaluation against Gold Standard BioCause Relations (Loaded in Section 4)
test_articles = set(df_aml_test.merge(df_cands_test_meta[['candidate_id', 'article_id']], on='candidate_id')['article_id'].unique())
gold_test_rels = df_gold_relations[df_gold_relations['article_id'].isin(test_articles)].copy()

G_gold = nx.DiGraph()
for _, r in gold_test_rels.iterrows():
    u = f"{r['section_id']}_{r['cause_id']}"
    v = f"{r['section_id']}_{r['effect_id']}"
    G_gold.add_edge(u, v)

def evaluate_graph_topology(G_g: nx.DiGraph, G_p: nx.DiGraph, label: str):
    gold_edges = set(G_g.edges())
    pred_edges = set(G_p.edges())
    tp = len(gold_edges & pred_edges)
    fp = len(pred_edges - gold_edges)
    fn = len(gold_edges - pred_edges)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {
        'Stage': label,
        'Gold Edges': len(gold_edges),
        'Pred Edges': len(pred_edges),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'Precision': p,
        'Recall': r,
        'F1': f1,
    }

eval_before = evaluate_graph_topology(G_gold, G_pred, "Before Cycle Pruning")
eval_after = evaluate_graph_topology(G_gold, G_dag, "After DAG Enforcement")

df_graph_eval = pd.DataFrame([eval_before, eval_after])
print("\n" + "=" * 80)
print("EDGE-LEVEL EVALUATION AGAINST GOLD DIRECTED RELATIONS (Test Split)")
print("=" * 80)
print(df_graph_eval[['Stage', 'Gold Edges', 'Pred Edges', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1']].to_string(index=False))
print("\nMethodological Note: Edge acyclicity guarantees formal structural consistency")
print("for downstream Bayesian networks, but is not alone an absolute proof of biological causal truth.")


--- Initial Graph Topology ---
  Nodes : 89
  Edges : 114
  Is DAG: False

--- Cycle Breaking Diagnostics & Audit Log ---
  Self-loops pruned             : 0
  Bidirectional conflicts pruned: 57
  Feedback cycle edges pruned   : 0
  Total edges removed           : 57
  Final DAG Nodes               : 89
  Final DAG Edges               : 57
  Acyclicity Guaranteed         : True

EDGE-LEVEL EVALUATION AGAINST GOLD DIRECTED RELATIONS (Test Split)
                Stage  Gold Edges  Pred Edges  TP  FP  FN  Precision  Recall       F1
 Before Cycle Pruning         100         114  44  70  56   0.385965    0.44 0.411215
After DAG Enforcement         100          57  36  21  64   0.631579    0.36 0.458599

Methodological Note: Edge acyclicity guarantees formal structural consistency
for downstream Bayesian networks, but is not alone an absolute proof of biological causal truth.


## 9. Probabilistic Bayesian Network Compilation & Interventional Queries

### 1. Structural Compilation of the Extracted DAG Topology
- The acyclic NetworkX DAG pruned from test predictions is converted into a `pgmpy.models.DiscreteBayesianNetwork(bn_edges)` instance.
- **Strict Methodological Distinction**: This compiled network captures **pure graph topology** (directed edges, parent-child structures, d-separation properties, and topological ordering). It does **NOT** possess empirically validated conditional probability distributions (CPDs), as observational text mining extracts causal assertions rather than population-level joint state frequency distributions.

### 2. Separate Illustrative Pearl $do$-Calculus Demonstration (Synthetic CPDs)
- To demonstrate formal causal reasoning ($do$-calculus vs. observational conditioning) using `pgmpy`, we define a separate, canonical 4-node regulatory pathway with synthetic, illustrative CPDs.
- This explicitly illustrates Pearl's graph mutilation ($do$-operator) versus Bayesian conditioning without falsely claiming that the large-scale mined graph has parameter-fitted CPDs.

In [8]:
# Section 8: Bayesian Network Compilation & Interventional Queries

# -------------------------------------------------------------------------
# Part 1: Structural Compilation of Extracted DAG Topology
# -------------------------------------------------------------------------
# The extracted acyclic DAG is compiled into a DiscreteBayesianNetwork preserving
# all nodes (including isolated nodes whose incident edges were pruned) and edges.
# Strict Methodological Boundary: Text mining alone yields structural causal assertions (edges),
# not joint state frequency distributions (CPDs). Parameterized CPDs are not claimed for the mined graph.

bn_model = dl_task4_utils.compile_discrete_bayesian_network(G_dag)
n_isolated = len(list(nx.isolates(bn_model)))
print(f"✓ Structurally compiled DiscreteBayesianNetwork: {len(bn_model.nodes())} nodes ({n_isolated} isolated nodes preserved), {len(bn_model.edges())} edges (Pure DAG topology).")

# -------------------------------------------------------------------------
# Part 2: Separate Illustrative Pearl do-Calculus Demonstration (Synthetic CPDs)
# -------------------------------------------------------------------------
# We construct a separate canonical 4-node biomedical pathway with synthetic CPDs
# strictly to demonstrate observational conditioning P(Y | X) vs causal graph surgery P(Y | do(X)).
demo_bn = DiscreteBayesianNetwork([
    ("Sensor_Kinase", "Response_Regulator"),
    ("Co_Activator", "Response_Regulator"),
    ("Response_Regulator", "Antibiotic_Resistance"),
])

cpd_sensor = TabularCPD(variable="Sensor_Kinase", variable_card=2, values=[[0.70], [0.30]], state_names={"Sensor_Kinase": ["0", "1"]})
cpd_coact = TabularCPD(variable="Co_Activator", variable_card=2, values=[[0.75], [0.25]], state_names={"Co_Activator": ["0", "1"]})
cpd_reg = TabularCPD(
    variable="Response_Regulator", variable_card=2,
    values=[
        [0.90, 0.40, 0.30, 0.05],
        [0.10, 0.60, 0.70, 0.95]
    ],
    evidence=["Sensor_Kinase", "Co_Activator"], evidence_card=[2, 2],
    state_names={"Response_Regulator": ["0", "1"], "Sensor_Kinase": ["0", "1"], "Co_Activator": ["0", "1"]}
)
cpd_res = TabularCPD(
    variable="Antibiotic_Resistance", variable_card=2,
    values=[
        [0.95, 0.15],
        [0.05, 0.85]
    ],
    evidence=["Response_Regulator"], evidence_card=[2],
    state_names={"Antibiotic_Resistance": ["0", "1"], "Response_Regulator": ["0", "1"]}
)

demo_bn.add_cpds(cpd_sensor, cpd_coact, cpd_reg, cpd_res)
assert demo_bn.check_model(), "Illustrative Bayesian network check_model failed"

infer_obs = VariableElimination(demo_bn)
q_base = infer_obs.query(["Antibiotic_Resistance"])
p_base = float(q_base.values[1])

# Observational query: P(Resistance=1 | Sensor=1)
q_obs = infer_obs.query(["Antibiotic_Resistance"], evidence={"Sensor_Kinase": "1"})
p_obs = float(q_obs.values[1])

# Interventional query via graph surgery: P(Resistance=1 | do(Response_Regulator=1))
interv_model = demo_bn.do(["Response_Regulator"])
infer_interv = VariableElimination(interv_model)
q_interv = infer_interv.query(["Antibiotic_Resistance"], evidence={"Response_Regulator": "1"})
p_interv = float(q_interv.values[1])

print("=" * 80)
print("ILLUSTRATIVE BAYESIAN NETWORK CAUSAL INFERENCE RESULTS (Synthetic CPDs)")
print("=" * 80)
print(f"1. Baseline P(Resistance = 1)                         : {p_base:.4f}")
print(f"2. Observational P(Resistance = 1 | Sensor = 1)       : {p_obs:.4f} (+{p_obs - p_base:.4f} associative shift)")
print(f"3. Interventional P(Resistance = 1 | do(Regulator=1)) : {p_interv:.4f} (pure causal effect via graph surgery)")
print("=" * 80)
print("Methodological Boundary: Text mining identifies DAG topology; numerical inference")
print("demonstrates do-calculus on synthetic CPDs, distinguishing observation from action.")

✓ Structurally compiled DiscreteBayesianNetwork: 89 nodes (0 isolated nodes preserved), 57 edges (Pure DAG topology).
ILLUSTRATIVE BAYESIAN NETWORK CAUSAL INFERENCE RESULTS (Synthetic CPDs)
1. Baseline P(Resistance = 1)                         : 0.3590
2. Observational P(Resistance = 1 | Sensor = 1)       : 0.6600 (+0.3010 associative shift)
3. Interventional P(Resistance = 1 | do(Regulator=1)) : 0.8500 (pure causal effect via graph surgery)
Methodological Boundary: Text mining identifies DAG topology; numerical inference
demonstrates do-calculus on synthetic CPDs, distinguishing observation from action.


## 10. Downstream Artifact Export for Task 5

We export standardized artifacts to `data/processed/` for downstream Task 5 evaluation and ablation:
1. `dl_biocause_hybrid_validation_predictions.csv`: 1,884 candidates with AML, DL, and Hybrid probabilities.
2. `dl_biocause_hybrid_test_predictions.csv`: 1,394 candidates with AML, DL, and Hybrid probabilities.
3. `dl_biocause_hybrid_positive_relations.csv`: 17-column `UnifiedRelationRecord` format.
4. `dl_biocause_hybrid_dag_edges.csv`: Final acyclic DAG edge list.
5. `dl_biocause_fusion_config.json`: Selected $\alpha^*$, threshold $\tau^*$, and optimization metadata.
6. `dl_biocause_graph_pruning_diagnostics.json`: Audit log of cycle-pruning and linguistic filtering.

In [9]:
# Section 9: Downstream Artifact Export for Task 5

# 1. Export hybrid validation predictions (1,884 rows)
val_hybrid_export = pd.DataFrame({
    'candidate_id': df_aml_val['candidate_id'].values,
    'true_label': val_y_true,
    'aml_prob': val_p_aml,
    'dl_prob': val_p_dl,
    'hybrid_prob': val_p_hybrid_best,
    'aml_pred': df_aml_val['aml_pred'].values,
    'dl_pred': df_dl_val['dl_pred'].values,
    'hybrid_pred': (val_p_hybrid_best >= FROZEN_TAU).astype(int),
    'split': 'validation',
})
val_exp_path = DATA_DIR / 'dl_biocause_hybrid_validation_predictions.csv'
dl_utils.export_predictions(val_hybrid_export, str(val_exp_path), format_type='csv')
dl_task4_utils.verify_prediction_export(
    filepath=val_exp_path,
    expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total'],
    split_name='validation',
    source_candidate_ids=df_aml_val['candidate_id'].values,
    source_true_labels=val_y_true,
    source_aml_probs=val_p_aml,
    source_dl_probs=val_p_dl,
    source_aml_preds=df_aml_val['aml_pred'].values,
    source_dl_preds=df_dl_val['dl_pred'].values,
    frozen_alpha=FROZEN_ALPHA,
    frozen_tau=FROZEN_TAU,
)
print(f"✓ Exported & verified hybrid validation predictions : {val_exp_path.relative_to(ROOT_DIR)} ({len(val_hybrid_export):,} rows)")

# 2. Export hybrid test predictions (1,394 rows)
test_hybrid_export = pd.DataFrame({
    'candidate_id': df_aml_test['candidate_id'].values,
    'true_label': test_y_true,
    'aml_prob': test_p_aml,
    'dl_prob': test_p_dl,
    'hybrid_prob': test_p_hybrid,
    'aml_pred': df_aml_test['aml_pred'].values,
    'dl_pred': df_dl_test['dl_pred'].values,
    'hybrid_pred': test_y_pred_hybrid,
    'sanitized_pred': df_test_full['sanitized_pred'].values,
    'split': 'test',
})
test_exp_path = DATA_DIR / 'dl_biocause_hybrid_test_predictions.csv'
dl_utils.export_predictions(test_hybrid_export, str(test_exp_path), format_type='csv')
dl_task4_utils.verify_prediction_export(
    filepath=test_exp_path,
    expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'],
    split_name='test',
    source_candidate_ids=df_aml_test['candidate_id'].values,
    source_true_labels=test_y_true,
    source_aml_probs=test_p_aml,
    source_dl_probs=test_p_dl,
    source_aml_preds=df_aml_test['aml_pred'].values,
    source_dl_preds=df_dl_test['dl_pred'].values,
    frozen_alpha=FROZEN_ALPHA,
    frozen_tau=FROZEN_TAU,
    include_sanitized_col=True,
)
print(f"✓ Exported & verified hybrid test predictions       : {test_exp_path.relative_to(ROOT_DIR)} ({len(test_hybrid_export):,} rows)")

# 3. Export unified relation records (17 columns)
rels_exp_path = DATA_DIR / 'dl_biocause_hybrid_positive_relations.csv'
dl_utils.export_predictions(df_unified_records, str(rels_exp_path), format_type='csv')
dl_task4_utils.verify_relation_export(
    filepath=rels_exp_path,
    expected_count=n_sanitized_pos,
    sanitized_candidate_ids=df_pos_relations["candidate_id"].tolist(),
    traceability_map=candidate_id_to_relation_id,
)
print(f"✓ Exported & verified unified positive relations    : {rels_exp_path.relative_to(ROOT_DIR)} ({len(df_unified_records):,} records)")

# 4. Export final DAG edge list
dag_edges_records = []
for u, v, d in G_dag.edges(data=True):
    dag_edges_records.append({
        'source': u,
        'target': v,
        'confidence': d.get('confidence', 1.0),
        'evidence_count': d.get('evidence_count', 1),
        'extraction_method': d.get('extraction_method', 'hybrid_probability_fusion'),
    })
df_dag_edges = pd.DataFrame(dag_edges_records)
dag_exp_path = DATA_DIR / 'dl_biocause_hybrid_dag_edges.csv'
dl_utils.export_predictions(df_dag_edges, str(dag_exp_path), format_type='csv')
print(f"✓ Exported final DAG edge list                      : {dag_exp_path.relative_to(ROOT_DIR)} ({len(df_dag_edges):,} edges)")

# 5. Export fusion configuration JSON
assert eval_hybrid_selected['Stream'].startswith("Hybrid Model"), (
    f"Exported model metrics identity is not the selected hybrid model! Got: {eval_hybrid_selected['Stream']}"
)

hybrid_test_metrics_export = {
    k: float(v) if isinstance(v, (int, float, np.floating)) else v
    for k, v in eval_hybrid_selected.items()
    if k != 'Confusion Matrix'
}

fusion_config = {
    'task': 'DL_04_hybrid_graph_integration',
    'selected_model': eval_hybrid_selected['Stream'],
    'alpha_star': float(FROZEN_ALPHA),
    'threshold_star': float(FROZEN_TAU),
    'optimization_objective': 'maximize_validation_positive_f1_with_macro_f1_tiebreak',
    'validation_metrics': {
        'positive_f1': float(best_val_f1_pos),
        'macro_f1': float(best_val_macro_f1),
    },
    'test_metrics': hybrid_test_metrics_export,
}
config_exp_path = DATA_DIR / 'dl_biocause_fusion_config.json'
with open(config_exp_path, 'w') as f:
    json.dump(fusion_config, f, indent=2)
print(f"✓ Exported fusion configuration JSON                : {config_exp_path.relative_to(ROOT_DIR)} (Model: {eval_hybrid_selected['Stream']})")

# 6. Export graph pruning diagnostics JSON
diagnostics_export = {
    'initial_candidate_edges': n_edges_before,
    'final_dag_edges': n_edges_after,
    'pruned_edges_count': len(audit_log),
    'pruning_breakdown': {
        'self_loops': n_self_loops,
        'bidirectional_conflicts': n_bidirectional,
        'directed_cycle_feedback': n_cycle_feedback,
    },
    'audit_log': audit_log,
    'edge_evaluation': {
        'before_pruning': eval_before,
        'after_pruning': eval_after,
    }
}
diag_exp_path = DATA_DIR / 'dl_biocause_graph_pruning_diagnostics.json'
with open(diag_exp_path, 'w') as f:
    json.dump(diagnostics_export, f, indent=2, default=lambda x: str(x) if hasattr(x, '__str__') else x)
dl_task4_utils.verify_dag_and_pruning_diagnostics(G_pred, G_dag, audit_log, diag_exp_path, dag_edges_filepath=dag_exp_path)
print(f"✓ Exported & verified graph pruning diagnostics     : {diag_exp_path.relative_to(ROOT_DIR)}")

✓ Exported & verified hybrid validation predictions : data/processed/dl_biocause_hybrid_validation_predictions.csv (1,884 rows)


✓ Exported & verified hybrid test predictions       : data/processed/dl_biocause_hybrid_test_predictions.csv (1,394 rows)


✓ Exported & verified unified positive relations    : data/processed/dl_biocause_hybrid_positive_relations.csv (114 records)
✓ Exported final DAG edge list                      : data/processed/dl_biocause_hybrid_dag_edges.csv (57 edges)
✓ Exported fusion configuration JSON                : data/processed/dl_biocause_fusion_config.json (Model: Hybrid Model (FROZEN alpha=0.00, tau=0.35))
✓ Exported & verified graph pruning diagnostics     : data/processed/dl_biocause_graph_pruning_diagnostics.json


In [10]:
# Section 10: Task Status and Verification Summary Gate
print("=" * 80)
print("TASK 4 COMPLETION SUMMARY & ACCEPTANCE CHECKLIST")
print("=" * 80)

# Verify exported artifacts content, schemas, row counts, and acyclicity
exports_verified = False
if (
    'val_exp_path' in globals() and val_exp_path.is_file() and
    'test_exp_path' in globals() and test_exp_path.is_file() and
    'rels_exp_path' in globals() and rels_exp_path.is_file() and
    'dag_exp_path' in globals() and dag_exp_path.is_file() and
    'config_exp_path' in globals() and config_exp_path.is_file() and
    'diag_exp_path' in globals() and diag_exp_path.is_file()
):
    try:
        dl_task4_utils.verify_prediction_export(
            filepath=val_exp_path,
            expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total'],
            split_name='validation',
            source_candidate_ids=df_aml_val['candidate_id'].values,
            source_true_labels=val_y_true,
            source_aml_probs=val_p_aml,
            source_dl_probs=val_p_dl,
            source_aml_preds=df_aml_val['aml_pred'].values,
            source_dl_preds=df_dl_val['dl_pred'].values,
            frozen_alpha=FROZEN_ALPHA,
            frozen_tau=FROZEN_TAU,
        )
        dl_task4_utils.verify_prediction_export(
            filepath=test_exp_path,
            expected_rows=dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'],
            split_name='test',
            source_candidate_ids=df_aml_test['candidate_id'].values,
            source_true_labels=test_y_true,
            source_aml_probs=test_p_aml,
            source_dl_probs=test_p_dl,
            source_aml_preds=df_aml_test['aml_pred'].values,
            source_dl_preds=df_dl_test['dl_pred'].values,
            frozen_alpha=FROZEN_ALPHA,
            frozen_tau=FROZEN_TAU,
            include_sanitized_col=True,
        )
        dl_task4_utils.verify_relation_export(
            filepath=rels_exp_path,
            expected_count=n_sanitized_pos,
            sanitized_candidate_ids=df_pos_relations["candidate_id"].tolist(),
            traceability_map=candidate_id_to_relation_id,
        )
        dl_task4_utils.verify_dag_and_pruning_diagnostics(G_pred, G_dag, audit_log, diag_exp_path, dag_edges_filepath=dag_exp_path)
        with open(config_exp_path) as f:
            cfg_exp = json.load(f)
        assert cfg_exp.get('selected_model', '').startswith("Hybrid Model")
        assert 'test_metrics' in cfg_exp and 'positive_f1' in cfg_exp.get('validation_metrics', {})
        exports_verified = True
    except Exception as e:
        print(f"Export content verification failed: {e}")
        exports_verified = False

ac01_input_contracts = bool(
    'df_aml_val' in globals() and 'df_dl_val' in globals() and
    len(df_aml_val) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total'] and
    len(df_dl_val) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['validation']['total'] and
    'df_aml_test' in globals() and 'df_dl_test' in globals() and
    len(df_aml_test) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'] and
    len(df_dl_test) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total']
)

ac02_alignment = bool(
    ac01_input_contracts and
    (df_aml_val['candidate_id'].values == df_dl_val['candidate_id'].values).all() and
    (df_aml_val['true_label'].values == df_dl_val['true_label'].values).all() and
    (df_aml_test['candidate_id'].values == df_dl_test['candidate_id'].values).all() and
    (df_aml_test['true_label'].values == df_dl_test['true_label'].values).all() and
    'expected_val_ids' in globals() and set(df_aml_val['candidate_id']) == expected_val_ids and
    'expected_test_ids' in globals() and set(df_aml_test['candidate_id']) == expected_test_ids
)

ac03_val_optimization = bool(
    'FROZEN_ALPHA' in globals() and 'FROZEN_TAU' in globals() and
    0.0 <= FROZEN_ALPHA <= 1.0 and 0.1 <= FROZEN_TAU <= 0.9 and
    'search_records' in globals() and len(search_records) > 0
)

ac04_test_frozen = bool(
    'test_p_hybrid' in globals() and 'test_y_pred_hybrid' in globals() and
    len(test_p_hybrid) == dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS['test']['total'] and
    'test_summary' in globals() and len(test_summary) >= 4 and
    'eval_hybrid_selected' in globals() and eval_hybrid_selected['Stream'].startswith("Hybrid Model")
)

ac05_pure_baselines = bool(
    'eval_aml' in globals() and 'eval_dl_documented' in globals() and 'eval_dl_default' in globals() and
    eval_aml['Threshold'] == 0.50 and eval_dl_documented['Threshold'] == 0.70
)

ac06_sanitized_filtering = bool(
    'df_test_full' in globals() and 'sanitized_pred' in df_test_full.columns and
    'diag_counts' in globals() and isinstance(diag_counts, dict) and
    diag_counts['initial_positive'] == (
        diag_counts['surviving_positive'] +
        diag_counts['rejected_negation_only'] +
        diag_counts['rejected_distance_only'] +
        diag_counts['rejected_both']
    ) and
    diag_counts['total_unique_rejected'] == (
        diag_counts['rejected_negation_only'] +
        diag_counts['rejected_distance_only'] +
        diag_counts['rejected_both']
    ) and
    diag_counts['initial_positive'] == diag_counts['surviving_positive'] + diag_counts['total_unique_rejected'] and
    diag_counts['surviving_positive'] == int((df_test_full['sanitized_pred'] == 1).sum()) and
    diag_counts['surviving_positive'] > 0 and
    diag_counts['total_unique_rejected'] >= 0
)

ac07_unified_records = bool(
    'df_unified_records' in globals() and
    list(df_unified_records.columns) == dl_utils.UNIFIED_RELATION_SCHEMA and
    len(df_unified_records) == n_sanitized_pos
)

ac08_dag_acyclicity = bool(
    'G_dag' in globals() and nx.is_directed_acyclic_graph(G_dag) and
    'audit_log' in globals() and len(audit_log) >= 0 and
    'eval_after' in globals() and all(0.0 <= eval_after[m] <= 1.0 for m in ['Precision', 'Recall', 'F1'])
)

ac09_bayesian_network = bool(
    'bn_model' in globals() and isinstance(bn_model, DiscreteBayesianNetwork) and
    set(bn_model.nodes()) == set(G_dag.nodes()) and
    set(bn_model.edges()) == set(G_dag.edges()) and
    nx.is_directed_acyclic_graph(bn_model) and
    'demo_bn' in globals() and isinstance(demo_bn, DiscreteBayesianNetwork) and
    demo_bn.check_model() and
    'q_obs' in globals() and hasattr(q_obs, 'values') and
    'q_interv' in globals() and hasattr(q_interv, 'values') and
    0.0 <= float(q_obs.values[1]) <= 1.0 and
    0.0 <= float(q_interv.values[1]) <= 1.0
)

ac10_exported_artifacts = exports_verified

acceptance_criteria = {
    "AC-01: Model-eligible prediction input contracts (1,884 val / 1,394 test)": ac01_input_contracts,
    "AC-02: Zero cross-stream candidate ID order & label discrepancies asserted": ac02_alignment,
    "AC-03: Validation-exclusive grid search with lexicographic tie-breaking": ac03_val_optimization,
    "AC-04: Test evaluated strictly ONCE using frozen alpha* and tau*": ac04_test_frozen,
    "AC-05: Pure AML (0.50) and Pure DL (0.70) baselines evaluated without retuning": ac05_pure_baselines,
    "AC-06: Non-overlapping linguistic rejection diagnostics verified": ac06_sanitized_filtering,
    "AC-07: Standardized 17-column UnifiedRelationRecord schema conformance": ac07_unified_records,
    "AC-08: Deterministic DAG acyclicity & edge evaluation audit completed": ac08_dag_acyclicity,
    "AC-09: DiscreteBayesianNetwork compilation & Pearl do-calculus demonstration": ac09_bayesian_network,
    "AC-10: Downstream artifact export and verified read-back integrity": ac10_exported_artifacts,
}

all_passed = all(acceptance_criteria.values())

for crit, status in acceptance_criteria.items():
    mark = "✓" if status else "✗"
    print(f"[{mark}] {crit}")

print("=" * 80)
if all_passed:
    print("ALL TASK 4 ACCEPTANCE CRITERIA VERIFIED SUCCESSFULLY.")
else:
    failed = [k for k, v in acceptance_criteria.items() if not v]
    print(f"TASK 4 INCOMPLETE: {len(failed)} criteria not satisfied:")
    for f_crit in failed:
        print(f"  - {f_crit}")
print("=" * 80)


TASK 4 COMPLETION SUMMARY & ACCEPTANCE CHECKLIST
[✓] AC-01: Model-eligible prediction input contracts (1,884 val / 1,394 test)
[✓] AC-02: Zero cross-stream candidate ID order & label discrepancies asserted
[✓] AC-03: Validation-exclusive grid search with lexicographic tie-breaking
[✓] AC-04: Test evaluated strictly ONCE using frozen alpha* and tau*
[✓] AC-05: Pure AML (0.50) and Pure DL (0.70) baselines evaluated without retuning
[✓] AC-06: Non-overlapping linguistic rejection diagnostics verified
[✓] AC-07: Standardized 17-column UnifiedRelationRecord schema conformance
[✓] AC-08: Deterministic DAG acyclicity & edge evaluation audit completed
[✓] AC-09: DiscreteBayesianNetwork compilation & Pearl do-calculus demonstration
[✓] AC-10: Downstream artifact export and verified read-back integrity
ALL TASK 4 ACCEPTANCE CRITERIA VERIFIED SUCCESSFULLY.
